In [18]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from src.data_loading import load_data, split_train_test_freq
from src.features import preparer, modalites_frequentes, regrouper, FORMULE_FREQUENCE_FINALE, appliquer_decisions_phase1
from src.graph import plot_freq_par_groupe, freq_par_groupe, plot_facteurs, plot_exposition_par_modalite
from src.metrics import comparer_tranches, seuil_aic_lr, seuil_bic_lr, seuil_test_lr, comparer_modeles_glm, lift_table, gini
from sklearn.metrics import mean_poisson_deviance
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
freq, sev = load_data()
train, test = split_train_test_freq(freq, test_size=0.2, random_state=42)

train = preparer(train)
test = preparer(test)

modalites = {}
ETIQUETTE_AUTRE = {"VehPower": "12+", "Region": "Autres", "VehBrand": "Autres"}

for col in ["Region", "VehBrand", "VehPower"]:
    modalites[col] = modalites_frequentes(train, col)
    train = regrouper(train, col, modalites[col], autre=ETIQUETTE_AUTRE[col])
    test  = regrouper(test,  col, modalites[col], autre=ETIQUETTE_AUTRE[col])


train = appliquer_decisions_phase1(train)
test = appliquer_decisions_phase1(test)

sev_train = sev.merge(train, on="IDpol", how="inner", validate="many_to_one")
sev_test  = sev.merge(test,  on="IDpol", how="inner", validate="many_to_one")

perdus = len(sev) - len(sev_train) - len(sev_test)
print(perdus)
print("Part du train : ", len(sev_train) / len(sev))


0
Part du train :  0.799122674330661


In [20]:
nb_lignes = sev_train.groupby("IDpol").size().rename("nb_lignes")
comp = train.set_index("IDpol").join(nb_lignes, how="inner")
print("Plus de lignes que ClaimNb :", (comp["nb_lignes"] > comp["ClaimNb"]).sum())
print("Moins de lignes que ClaimNb :", (comp["nb_lignes"] < comp["ClaimNb"]).sum())

Plus de lignes que ClaimNb : 5
Moins de lignes que ClaimNb : 1


On constat que 5 polices ont plus de ligne dans `sev` que dnas la variable `ClaimNB` de `freq`. Ces polices sont surement celle dont on a plafonné `ClaimNB` dans la phase d'exploration mais dont les montants correspondant sont resté dans `sev`. Le plafonnement de `ClaimNB` a été fait, il n'y a rien à faire.

1 police à moins de ligne dans `sev` que dans `ClaimNB` de `freq`. L'estimation du coût moyen porte sur les sinistres chiffrée uniquement, il n'y a donc rien a faire pour cette police.

Il est nécessaire pour l'estimation du coût moyen de ne pas inclure les sinistres de valeurs extrême car ils fausseraient l'estimation et pourraient conduire à des primes très élevées pour certaine tranche à faible exposition. Les évênement extrêmes relevent du hasard et ne sont pas représentatif d'un profil de risque. Il est donc nécessaire de les exclure pour l'estimation du coût moyen.

La solution est de plafonner chaque sinistre à un seuil et de redistribuer l'excédent sous forme d'un facteur de charge uniforme.

In [41]:
lignes = []
for q in [0.95, 0.975, 0.99, 0.995, 0.999]:
    s = sev_train["ClaimAmount"].quantile(q)
    exces = (sev_train["ClaimAmount"] - s).clip(lower=0)
    lignes.append({
        "quantile" : q,
        "seuil" : s, 
        "nb_au_dessus" : (sev_train["ClaimAmount"] > s).sum(),
        "part_charge_excedent" : exces.sum() / sev_train["ClaimAmount"].sum(),
        "kappa" : exces.sum() / sev_train["ClaimAmount"].clip(upper=s).sum()})

seuils = pd.DataFrame(lignes).round(3)
seuils



,quantile,seuil,nb_au_dessus,part_charge_excedent,kappa
0,0.950,4760.228,1057,0.428,0.748
1,0.975,8101.782,529,0.377,0.605
2,0.990,16351.621,212,0.322,0.474
3,0.995,33503.045,106,0.271,0.371
4,0.999,150131.075,22,0.169,0.204


Le choix d'un seuil au quantile 0.99 me semble approprié. J'ai peur qu'avec des seuils plus hauts, les valeurs extrêmes (au dessus de 350000 euros) faussent l'estimation du coût moyen pour des profils et modalités à faible exposition.

In [ ]:
top = sev_train["ClaimAmount"].nlargest(10)
print(top)
print("Part de la charge porté par les 10 plus gros : ", sev_train["ClaimAmount"].nlargest(10).sum() / sev_train["ClaimAmount"].sum())

7781     4075400.56
12601    1403057.40
9286     1301172.60
14387     774411.50
2315      390742.27
7975      307096.42
14645     281897.49
16274     276643.00
6338      255013.03
11073     241171.16
Name: ClaimAmount, dtype: float64
Part de la charge porté par les 10 plus gros :  0.19050017929893187


In [40]:
print(sev_train["ClaimAmount"].value_counts().head(10))

ClaimAmount
1204.00    3833
1128.12    2447
1172.00    1678
1128.00     666
602.00      339
1320.00     133
556.14      117
586.00       84
564.06       73
1500.00      71
Name: count, dtype: int64


J'ai remarqué que 602 et 586 et 564.06 sont les moitiés respective de 1204, 1172 et 1128. Je n'ai pas encore trop parlé des forfaits en cours mais j'imagine que ce sont des cas où l'assureur ne rembourse que la moitié du forfait. On va donc les intégrer à la liste des forfaits et les traiter comme tel.

In [42]:
forfaits = [1204, 1128.12, 1172, 1128, 602, 586, 564.06]
est_forfait = sev_train["ClaimAmount"].round(2).isin(forfaits)
print("Part des lignes qui sont des forfaits : ", est_forfait.sum() / len(sev_train))
print("Part de la charge que représente les forfaits : ", sev_train["ClaimAmount"][est_forfait].sum() / sev_train["ClaimAmount"].sum())

Part des lignes qui sont des forfaits :  0.43157296990346394
Part de la charge que représente les forfaits :  0.21263120406390532


In [39]:
print("Sous 40 € :", (sev_train["ClaimAmount"] < 40).sum(), "— minimum :", x.min())

Sous 40 € : 215 — minimum : 1.0


In [43]:
SEUIL_GRAVES = sev_train["ClaimAmount"].quantile(0.99)

sev_train["montant_ecrete"] = sev_train["ClaimAmount"].clip(upper=SEUIL_GRAVES)
sev_test["montant_ecrete"] = sev_test["ClaimAmount"].clip(upper=SEUIL_GRAVES)

excedent = (sev_train["ClaimAmount"] - SEUIL_GRAVES).clip(lower=0).sum()
K_GRAVES = excedent / sev_train["montant_ecrete"].sum()

print(f"Seuil : {SEUIL_GRAVES:.0f} €   k : {K_GRAVES:.4f}")

Seuil : 16352 €   k : 0.4743
